### Notebook 1 — Exploration et prise en main de Spark

In [2]:
from pyspark.sql import SparkSession  # type: ignore[reportMissingImports]
from pyspark.sql.functions import min, max, avg, stddev, lit
#Pandas
import pandas as pd
import time
#Pyspark

In [3]:
spark = SparkSession.builder.appName("TradeCorp ETL notebook 1").getOrCreate()
spark

In [4]:
DATA_PATH = "/home/jovyan/data/source_data"   # adapte selon ton volume Docker

# Lecture des fichiers CSV
df_customers      = spark.read.csv(f"{DATA_PATH}/customers.csv", header=True, inferSchema=True)
df_orders         = spark.read.csv(f"{DATA_PATH}/orders.csv", header=True, inferSchema=True)
df_order_details  = spark.read.csv(f"{DATA_PATH}/order_details.csv", header=True, inferSchema=True)
df_products       = spark.read.csv(f"{DATA_PATH}/products.csv", header=True, inferSchema=True)
df_categories     = spark.read.csv(f"{DATA_PATH}/categories.csv", header=True, inferSchema=True)
df_suppliers      = spark.read.csv(f"{DATA_PATH}/suppliers.csv", header=True, inferSchema=True)
df_employers      = spark.read.csv(f"{DATA_PATH}/employees.csv", header=True, inferSchema=True)
df_shippers       = spark.read.csv(f"{DATA_PATH}/shippers.csv", header=True, inferSchema=True)

# Affichage des 5 premières lignes pour chaque DataFrame
dfs = {
    "customers": df_customers,
    "orders": df_orders,
    "order_details": df_order_details,
    "products": df_products,
    "categories": df_categories,
    "suppliers": df_suppliers,
    "employers": df_employers,
    "shippers": df_shippers
}

In [5]:
#Lire les 8 fichiers
for name, df in dfs.items():
    print(f"\n=== {name.upper()} : 2 premières lignes ===")
    df.printSchema()


=== CUSTOMERS : 2 premières lignes ===
root
 |-- customer_id: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- contact_name: string (nullable = true)
 |-- contact_title: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- country: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- fax: string (nullable = true)


=== ORDERS : 2 premières lignes ===
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- required_date: date (nullable = true)
 |-- shipped_date: date (nullable = true)
 |-- ship_via: integer (nullable = true)
 |-- freight: double (nullable = true)
 |-- ship_name: string (nullable = true)
 |-- ship_address: string (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- ship_

In [6]:
# Tableau récapitulatif Python

summary = []

for name, df in dfs.items():
    count = df.count()
    summary.append((name, count))
    print(f"{name}: {count} lignes")

customers: 91 lignes
orders: 830 lignes
order_details: 2155 lignes
products: 77 lignes
categories: 8 lignes
suppliers: 29 lignes
employers: 9 lignes
shippers: 6 lignes


##### A6 Statistiques descriptives

In [7]:
def compute_stats(df, name):
    print(f"\n===== Statistiques pour {name} =====")

    # Colonnes numériques uniquement
    numeric_cols = [c for c, t in df.dtypes if t in ("int", "double", "float", "bigint")]

    if not numeric_cols:
        print("Aucune colonne numérique trouvée.")
        return None

    # Expressions de stats
    stats_exprs = []
    for c in numeric_cols:
        stats_exprs.extend(
            [
            min(c).alias(f"{c}_min"),
            max(c).alias(f"{c}_max"),
            avg(c).alias(f"{c}_mean"),
            stddev(c).alias(f"{c}_stddev")
        ])

    stats_df = df.select(*stats_exprs)
    stats_df = stats_df.withColumn("dataset", lit(name))

    stats_df.show(truncate=False)
    return stats_df


In [8]:
stats_orders = compute_stats(df_orders, "orders")


===== Statistiques pour orders =====
+------------+------------+-------------+-----------------+---------------+---------------+-----------------+------------------+------------+------------+------------------+------------------+-----------+-----------+-----------------+------------------+-------+
|order_id_min|order_id_max|order_id_mean|order_id_stddev  |employee_id_min|employee_id_max|employee_id_mean |employee_id_stddev|ship_via_min|ship_via_max|ship_via_mean     |ship_via_stddev   |freight_min|freight_max|freight_mean     |freight_stddev    |dataset|
+------------+------------+-------------+-----------------+---------------+---------------+-----------------+------------------+------------+------------+------------------+------------------+-----------+-----------+-----------------+------------------+-------+
|10248       |11077       |10662.5      |239.7446558319914|1              |9              |4.403614457831325|2.4996475400823885|1           |3           |2.0072289156626506|0.7

In [9]:
stats_products = compute_stats(df_products, "products")


===== Statistiques pour products =====
+--------------+--------------+---------------+------------------+---------------+---------------+-----------------+------------------+---------------+---------------+-----------------+------------------+--------------+--------------+------------------+-----------------+------------------+------------------+-------------------+---------------------+------------------+------------------+-------------------+---------------------+-----------------+-----------------+------------------+--------------------+----------------+----------------+-------------------+-------------------+--------+
|product_id_min|product_id_max|product_id_mean|product_id_stddev |supplier_id_min|supplier_id_max|supplier_id_mean |supplier_id_stddev|category_id_min|category_id_max|category_id_mean |category_id_stddev|unit_price_min|unit_price_max|unit_price_mean   |unit_price_stddev|units_in_stock_min|units_in_stock_max|units_in_stock_mean|units_in_stock_stddev|units_on_order_min

#### A9 Comaparer Pandas vs Pysspark

In [10]:
start = time.time()
pdf_orders = pd.read_csv("/home/jovyan/data/source_data/orders.csv")
end = time.time()

pandas_time = end - start
print(f"Temps Pandas : {pandas_time:.4f} secondes")


Temps Pandas : 0.0355 secondes


In [11]:
spark = SparkSession.builder.getOrCreate()

start = time.time()
df_orders = spark.read.csv("/home/jovyan/data/source_data/orders.csv", header=True, inferSchema=True)
df_orders.count()   # force Spark à lire le fichier
end = time.time()

spark_time = end - start
print(f"Temps Spark : {spark_time:.4f} secondes")



Temps Spark : 0.5608 secondes


In [12]:
summary = [
    ("pandas", pandas_time),
    ("spark", spark_time)
]

summary_df = spark.createDataFrame(summary, ["methode", "temps_seconds"])
summary_df.show()


+-------+-------------------+
|methode|      temps_seconds|
+-------+-------------------+
| pandas|0.03551959991455078|
|  spark| 0.5607864856719971|
+-------+-------------------+



#### A10 Colonnes et types0

In [13]:


cols_to_cast = []

for col, dtype in df_orders.dtypes:
    # Colonnes numériques mal typées
    if dtype == "string":
        # Test : est-ce que la colonne contient uniquement des chiffres ?
        if df_orders.select(col).rdd.map(lambda x: x[0]).filter(lambda x: x is not None).filter(lambda x: not x.isdigit()).count() == 0:
            cols_to_cast.append((col, "int"))
        
        # Test : est-ce que la colonne ressemble à une date ?
        elif df_orders.select(col).rdd.map(lambda x: x[0]).filter(lambda x: x is not None).filter(lambda x: "-" in x).count() > 0:
            cols_to_cast.append((col, "date"))

print("\nColonnes qui nécessitent un cast :")
print(cols_to_cast)


Colonnes qui nécessitent un cast :
[('ship_name', 'date'), ('ship_address', 'date'), ('ship_postal_code', 'date')]


#### fin notebook 1